<a href="https://colab.research.google.com/github/LuanLindolfo/regression_tomate/blob/main/tomate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import os
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
# ==========================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
# ==========================================
print("⏳ Baixando o dataset diretamente do Kaggle...")

# Download automatizado da pasta do dataset da Agricultura Digital (Sem chaves!)
caminho_pasta = kagglehub.dataset_download("mathurinache/agricultura-digital")
print(f"✅ Pasta baixada em: {caminho_pasta}")

# Apontando para o arquivo específico do tomate
caminho_arquivo = os.path.join(caminho_pasta, "dataset_tomate.csv")

print("\n⚙️ Carregando e preparando os dados no Pandas...")
# Carrega o dataset
df = pd.read_csv(caminho_arquivo)

⏳ Baixando o dataset diretamente do Kaggle...


100%|██████████| 15.8k/15.8k [00:00<00:00, 27.2MB/s]

Extracting files...
✅ Pasta baixada em: /root/.cache/kagglehub/datasets/mathurinache/agricultura-digital/versions/1

⚙️ Carregando e preparando os dados no Pandas...


In [7]:
# Separação entre Previsores (X) e Variável Alvo/Objetivo (Y)
# X: Leituras do dia 01 (Colunas 7, 8 e 9: NDVI, SAVI, GNDVI)
X = df.iloc[:, 7:10].values

# Y: Resultado do dia 28 (Coluna 2: NDVI_d28)
Y = df.iloc[:, 2].values

# Padronização dos dados (Crucial para KNN, SVM e Redes Neurais)
scaler = StandardScaler()
X_padronizado = scaler.fit_transform(X)

# Divisão em Treino (80%) e Teste (20%)
X_treino, X_teste, Y_treino, Y_teste = train_test_split(
    X_padronizado, Y, test_size=0.20, random_state=42
)

print(f"📈 Registros para treinamento: {X_treino.shape[0]}")
print(f"📉 Registros para teste: {X_teste.shape[0]}")

📈 Registros para treinamento: 105
📉 Registros para teste: 27


In [9]:
# ==========================================
# TESTE 1: Regressão Linear Múltipla
# ==========================================
reg_linear = LinearRegression()
# Ajustado para usar as variáveis definidas na célula Mwyj5VLdISuF
reg_linear.fit(X_treino, Y_treino)
prev_linear = reg_linear.predict(X_teste)
mae_linear = mean_absolute_error(Y_teste, prev_linear)
print(f"MAE Regressão Linear: {mae_linear}")

MAE Regressão Linear: 0.009978721106137865


In [11]:
# ==========================================
# TESTE 2: Árvore de Decisão
# ==========================================
reg_arvore = DecisionTreeRegressor()
# Ajustado para usar as variáveis em português definidas anteriormente
reg_arvore.fit(X_treino, Y_treino)
prev_arvore = reg_arvore.predict(X_teste)
mae_arvore = mean_absolute_error(Y_teste, prev_arvore)
print(f"MAE Árvore de Decisão: {mae_arvore}")

MAE Árvore de Decisão: 0.012200629444444453


In [13]:
# ==========================================
# TESTE 3: Random Forest
# ==========================================
reg_rf = RandomForestRegressor(n_estimators=100)
# Ajustado para usar as variáveis em português
reg_rf.fit(X_treino, Y_treino)
prev_rf = reg_rf.predict(X_teste)
mae_rf = mean_absolute_error(Y_teste, prev_rf)
print(f"MAE Random Forest: {mae_rf}")

MAE Random Forest: 0.010447461815926049


In [15]:
# ==========================================
# TESTE 4: SVR (Máquinas de Vetores de Suporte)
# ==========================================
# Como X_treino e X_teste já foram padronizados na preparação,
# precisamos apenas escalonar o Y para o SVR funcionar corretamente.

scaler_y = StandardScaler()
Y_treino_scaled = scaler_y.fit_transform(Y_treino.reshape(-1, 1))

reg_svr = SVR(kernel='rbf')
reg_svr.fit(X_treino, Y_treino_scaled.ravel())

# Predição e inversão da escala do resultado
prev_svr_scaled = reg_svr.predict(X_teste).reshape(-1, 1)
prev_svr = scaler_y.inverse_transform(prev_svr_scaled)

mae_svr = mean_absolute_error(Y_teste, prev_svr)
print(f"MAE SVR: {mae_svr}")

MAE SVR: 0.011320986848633696


In [17]:
# ==========================================
# TESTE 5: Redes Neurais (MLP)
# ==========================================
# Usando o scaler_y definido no teste anterior para manter a consistência
# Se a célula do SVR não foi executada, definimos o scaler_y aqui:
if 'scaler_y' not in locals():
    scaler_y = StandardScaler()
    Y_treino_scaled = scaler_y.fit_transform(Y_treino.reshape(-1, 1))
else:
    Y_treino_scaled = scaler_y.transform(Y_treino.reshape(-1, 1))

reg_mlp = MLPRegressor(max_iter=1000, hidden_layer_sizes=(9,9), random_state=42)
reg_mlp.fit(X_treino, Y_treino_scaled.ravel())

# Predição e inversão da escala
prev_mlp_scaled = reg_mlp.predict(X_teste).reshape(-1, 1)
prev_mlp = scaler_y.inverse_transform(prev_mlp_scaled)

mae_mlp = mean_absolute_error(Y_teste, prev_mlp)
print(f"MAE Redes Neurais: {mae_mlp}")

MAE Redes Neurais: 0.010380402079582886


O MAE (Mean Absolute Error) é uma métrica que mede a média das distâncias entre as previsões do modelo e os valores reais.

Em termos simples: Ele diz, em média, o quanto o modelo 'errou o alvo'.

Quanto menor, melhor.